# Daily Challenge: Custom Attention Mechanism & SMS Spam Classification
## Week 7 — Day 1 | DI GenAI & Machine Learning Bootcamp 2026

**Objective:** Build a custom attention mechanism from scratch, compare it to fine-tuned GPT-2, and apply both to real-world SMS spam detection.

**Parts:**
1. Setup & Data Loading
2. Tokenization Setup
3. Pre-trained GPT-2 Model Setup
4. Custom Attention Implementation
5. Metrics & Evaluation
6. Reflection Questions

> **Run on Google Colab** — GPU recommended.

## Part 1 — Setup & Data Loading

In [ ]:
# 1. Install required packages
!pip install --quiet datasets evaluate transformers[sentencepiece]

In [ ]:
# 2. Import necessary modules
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from datasets import Dataset as HFDataset
import warnings
warnings.filterwarnings("ignore")

print(f"PyTorch {torch.__version__} ✓")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")

In [ ]:
# 3. Load & inspect the SMS Spam dataset
df = pd.read_parquet("hf://datasets/ucirvine/sms_spam/plain_text/train-00000-of-00001.parquet")

# Convert df to Hugging Face Dataset
hf_dataset = HFDataset.from_pandas(df)

# Split the data — 4,000 training / 1,000 validation
train_ds = hf_dataset.select(range(4000))
val_ds   = hf_dataset.select(range(4000, 5000))

print(f"Dataset loaded ✓")
print(f"  Total samples  : {len(hf_dataset):,}")
print(f"  Training       : {len(train_ds):,}")
print(f"  Validation     : {len(val_ds):,}")
print(f"  Columns        : {hf_dataset.column_names}")
print(f"\nClass distribution (full dataset):")
print(df['label'].value_counts().rename({0: 'ham (0)', 1: 'spam (1)'}))
print("\nFirst 5 rows:")
display(df.head())

## Part 2 — Tokenization Setup

In [ ]:
from transformers import GPT2Tokenizer

# 1. Initialize the tokenizer
model_name = "gpt2"
tokenizer  = GPT2Tokenizer.from_pretrained(model_name)

# GPT-2 has no pad token by default — set it to eos
tokenizer.pad_token = tokenizer.eos_token

print(f"Tokenizer loaded ✓")
print(f"  Vocab size     : {tokenizer.vocab_size:,}")
print(f"  Pad token      : '{tokenizer.pad_token}'  (id={tokenizer.pad_token_id})")
print(f"  EOS token      : '{tokenizer.eos_token}'  (id={tokenizer.eos_token_id})")

In [ ]:
# 2. Create tokenization function
def tokenize_fn(examples):
    return tokenizer(
        examples["sms"],          # text column name
        padding="max_length",     # pad shorter sequences to max_length
        truncation=True,          # truncate sequences longer than max_length
        max_length=64             # fixed sequence length for batch processing
    )

# Apply tokenization to both datasets
train_tok = train_ds.map(tokenize_fn, batched=True)
val_tok   = val_ds.map(tokenize_fn,   batched=True)

print("Tokenization applied ✓")
print(f"  Train features : {list(train_tok.features.keys())}")
sample = train_tok[0]
print(f"  Sample input_ids length : {len(sample['input_ids'])}")
print(f"  Sample label            : {sample['label']}  ({'spam' if sample['label'] == 1 else 'ham'})")

## Part 3 — Pre-trained GPT-2 Model Setup

In [ ]:
from transformers import GPT2ForSequenceClassification

# Initialize GPT-2 for sequence classification (binary: ham=0, spam=1)
model = GPT2ForSequenceClassification.from_pretrained(
    "gpt2",                              # model name
    num_labels=2,                        # binary classification
    pad_token_id=tokenizer.eos_token_id  # set pad token id
)
model = model.to(device)

print(f"GPT-2 classification model loaded ✓")
print(f"  Parameters    : {model.num_parameters():,}")
print(f"  num_labels    : {model.config.num_labels}")
print(f"  pad_token_id  : {model.config.pad_token_id}")

In [ ]:
from transformers import Trainer, TrainingArguments

# Fine-tune GPT-2 on the SMS spam task (3 epochs, fast for Colab)
training_args = TrainingArguments(
    output_dir="./gpt2_sms",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    evaluation_strategy="epoch",
    save_strategy="no",
    logging_steps=50,
    learning_rate=2e-5,
    weight_decay=0.01,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
)

print("Fine-tuning GPT-2 on SMS Spam dataset...")
trainer.train()
print("Fine-tuning complete ✓")

## Part 4 — Custom Attention Implementation

In [ ]:
# 1. Implement the Attention class (scaled dot-product attention)
class Attention(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        # Scaling factor: 1/sqrt(d_k) — prevents vanishing gradients from large dot products
        self.scale = embed_dim ** -0.5

    def forward(self, query, key, value, mask=None):
        # Attention scores: Q × K^T / sqrt(d_k)
        # query: (batch, seq, d)  key: (batch, seq, d)  → scores: (batch, seq, seq)
        scores = torch.matmul(query, key.transpose(-2, -1)) * self.scale

        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        # Softmax over last dimension → attention weights sum to 1 per query position
        attn = F.softmax(scores, dim=-1)

        # Weighted sum of values
        return torch.matmul(attn, value), attn


print("Attention class defined ✓")

# Quick sanity check
batch, seq, d = 2, 8, 64
q = torch.randn(batch, seq, d)
attn_layer = Attention(d)
out, weights = attn_layer(q, q, q)
print(f"  Input  : {q.shape}")
print(f"  Output : {out.shape}  (same shape as input — self-attention)")
print(f"  Weights: {weights.shape}  (batch, seq, seq) — attention map")

In [ ]:
# 2. Build the attention-based classifier
class SimpleAttentionClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.attn      = Attention(embed_dim)
        self.fc        = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        embed = self.embedding(x)                    # (batch, seq, embed_dim)
        attn_output, _ = self.attn(embed, embed, embed)  # self-attention: Q=K=V=embed
        pooled = attn_output.mean(dim=1)             # mean over sequence → (batch, embed_dim)
        return self.fc(pooled)                       # (batch, num_classes)


print("SimpleAttentionClassifier defined ✓")

In [ ]:
# 3. Preprocess data for the custom model
def preprocess_for_attention(example):
    tokens = tokenizer.encode(
        example["sms"],
        max_length=64,
        truncation=True,
        padding="max_length"
    )
    return {"input_ids": tokens, "label": example["label"]}

train_ds_attn = train_ds.map(preprocess_for_attention)
val_ds_attn   = val_ds.map(preprocess_for_attention)

print("Preprocessing for custom model applied ✓")
print(f"  Sample input_ids length : {len(train_ds_attn[0]['input_ids'])}")


# 4. Create PyTorch DataLoader
class SMSDataset(Dataset):
    def __init__(self, hf_dataset):
        self.data = hf_dataset

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            'input_ids': torch.tensor(item["input_ids"], dtype=torch.long),
            'label':     torch.tensor(item["label"],     dtype=torch.long)
        }

train_loader = DataLoader(SMSDataset(train_ds_attn), batch_size=32, shuffle=True)
val_loader   = DataLoader(SMSDataset(val_ds_attn),   batch_size=32)

print(f"DataLoaders ready ✓")
print(f"  Train batches : {len(train_loader)}")
print(f"  Val batches   : {len(val_loader)}")

In [ ]:
# 5. Train the custom attention model
vocab_size  = tokenizer.vocab_size   # 50,257 for GPT-2
embed_dim   = 64
num_classes = 2                      # binary: ham / spam

attn_model = SimpleAttentionClassifier(vocab_size, embed_dim, num_classes).to(device)
optimizer  = torch.optim.Adam(attn_model.parameters(), lr=1e-3)
criterion  = nn.CrossEntropyLoss()

EPOCHS = 5
train_losses = []

print(f"Training custom attention model ({EPOCHS} epochs)...")
for epoch in range(1, EPOCHS + 1):
    attn_model.train()
    epoch_loss = 0.0
    for batch in train_loader:
        inputs = batch['input_ids'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = attn_model(inputs)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)
    print(f"  Epoch {epoch}/{EPOCHS}  Loss: {avg_loss:.4f}")

print("\nCustom Attention model trained ✓")
print(f"Final batch loss: {loss.item():.4f}")

# Plot training loss
plt.figure(figsize=(8, 4))
plt.plot(range(1, EPOCHS+1), train_losses, 'o-', color='#4e91d4', lw=2)
plt.title('Custom Attention Model — Training Loss', fontweight='bold')
plt.xlabel('Epoch'); plt.ylabel('Cross-Entropy Loss')
plt.grid(True, alpha=0.3); plt.tight_layout()
plt.savefig('dc_attn_training_loss.png', dpi=120, bbox_inches='tight')
plt.show()
print("Plot saved ✓")

## Part 5 — Metrics & Evaluation

In [ ]:
import evaluate

# 1. Load evaluation metrics
accuracy  = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall    = evaluate.load("recall")
f1        = evaluate.load("f1")

def compute_metrics(predictions, references):
    return {
        "accuracy":  accuracy.compute( predictions=predictions, references=references)["accuracy"],
        "precision": precision.compute(predictions=predictions, references=references)["precision"],
        "recall":    recall.compute(   predictions=predictions, references=references)["recall"],
        "f1":        f1.compute(       predictions=predictions, references=references)["f1"]
    }

print("Evaluation metrics loaded ✓")

In [ ]:
# 2a. Evaluate GPT-2 model
print("📊 Evaluating GPT-2 model...")
gpt2_preds, gpt2_labels = [], []
model.eval()
for ex in val_tok:
    inputs = torch.tensor(ex['input_ids']).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(inputs).logits
    pred = torch.argmax(logits, dim=-1).cpu().item()
    gpt2_preds.append(pred)
    gpt2_labels.append(ex['label'])

gpt2_metrics = compute_metrics(gpt2_preds, gpt2_labels)
print("GPT-2 Metrics:", {k: round(v, 4) for k, v in gpt2_metrics.items()})

# 2b. Evaluate Custom Attention model
print("\n📊 Evaluating Custom Attention model...")
attn_preds, attn_labels = [], []
attn_model.eval()
for batch in val_loader:
    inputs = batch['input_ids'].to(device)
    labels = batch['label'].to(device)
    with torch.no_grad():
        outputs = attn_model(inputs)
        preds   = torch.argmax(outputs, dim=1)
    attn_preds.extend(preds.cpu().tolist())
    attn_labels.extend(labels.cpu().tolist())

attn_metrics = compute_metrics(attn_preds, attn_labels)
print("Attention Model Metrics:", {k: round(v, 4) for k, v in attn_metrics.items()})

In [ ]:
# Side-by-side comparison plot
metrics_names = ['accuracy', 'precision', 'recall', 'f1']
gpt2_vals  = [gpt2_metrics[m]  for m in metrics_names]
attn_vals  = [attn_metrics[m]  for m in metrics_names]

x = np.arange(len(metrics_names))
width = 0.35
fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, gpt2_vals, width, label='GPT-2 (fine-tuned)', color='#4e91d4')
bars2 = ax.bar(x + width/2, attn_vals, width, label='Custom Attention',   color='#e05c5c')

for bar, v in zip(bars1, gpt2_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005, f'{v:.3f}',
            ha='center', va='bottom', fontsize=9)
for bar, v in zip(bars2, attn_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005, f'{v:.3f}',
            ha='center', va='bottom', fontsize=9)

ax.set_xticks(x); ax.set_xticklabels([m.capitalize() for m in metrics_names])
ax.set_ylim(0, 1.1); ax.set_ylabel('Score')
ax.set_title('GPT-2 vs Custom Attention — SMS Spam Classification', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('dc_comparison_metrics.png', dpi=120, bbox_inches='tight')
plt.show()
print("Plot saved ✓")

# Summary table
print(f"\n{'='*55}")
print(f"  {'Metric':<12} {'GPT-2':>10} {'Custom Attn':>14}")
print(f"{'='*55}")
for m in metrics_names:
    diff = gpt2_metrics[m] - attn_metrics[m]
    winner = '← GPT-2' if diff > 0 else '← Custom'
    print(f"  {m.capitalize():<12} {gpt2_metrics[m]:>10.4f} {attn_metrics[m]:>14.4f}  {winner}")
print(f"{'='*55}")

## Part 6 — Reflection Questions

### Q1: What are the roles of Query, Key, and Value in the attention mechanism?

In scaled dot-product attention, each input token is projected into three distinct roles:

- **Query (Q):** Represents what a token is *looking for*. The query of a given position asks "which other tokens are relevant to me?" It drives the search.
- **Key (K):** Represents what a token *advertises about itself*. Keys are matched against queries to determine relevance scores. A high Q·K dot product means "this position is highly relevant to what the query is looking for."
- **Value (V):** Represents the actual *content* a token contributes. Once relevance scores are computed (via softmax), values from highly-attended positions are summed with high weight. The output is a weighted blend of values — information aggregated based on relevance.

Intuitively: the query searches, the key indexes, and the value delivers.

---

### Q2: Why do we use a scaling factor (1/√d_k) in dot-product attention?

As the embedding dimension `d_k` grows larger, the dot products Q·K^T tend to grow in magnitude proportionally to √d_k (since they are sums of d_k products of independent random variables). This pushes the softmax into **saturation regions** — where the output probabilities become extremely peaked (close to 0 or 1) and gradients vanish nearly to zero, stalling training.

Dividing by √d_k **normalizes the variance** of the dot products back to ~1 regardless of dimensionality, keeping the softmax in a well-behaved regime where gradients flow effectively. Without scaling, the model would struggle to train at higher dimensions.

---

### Q3: How does self-attention differ from traditional sequence models like RNNs?

| Aspect | RNN | Self-Attention |
|---|---|---|
| **Processing** | Sequential (token by token) | Fully parallel (all tokens at once) |
| **Long-range dependencies** | Struggle — information must pass through many recurrent steps, fading with distance | Direct — every token can attend to every other token in O(1) steps |
| **Computational complexity** | O(n) sequential steps | O(n²) per layer but fully parallelizable on GPUs |
| **Training** | Prone to vanishing/exploding gradients over long sequences | More stable — direct gradient paths from output to any token |
| **Context window** | Limited by gradient propagation over time | Limited only by positional encoding and memory |

Self-attention is why Transformers train faster and generalize better on long-range tasks.

---

### Q4: Performance Analysis

**GPT-2 (fine-tuned)** significantly outperforms the custom attention model across all metrics. This is expected: GPT-2 brings 117M parameters of pretrained language knowledge — it already understands grammar, word meaning, and context from large-scale pretraining. Fine-tuning only needs to redirect this knowledge toward spam/ham discrimination.

The **custom attention model**, with only an embedding layer + a single attention layer + a linear head (~3M parameters, trained from scratch), learns much more limited representations in just 5 epochs. It captures some token co-occurrence patterns but lacks the deep, contextual representations that come from pretraining.

**Possible improvements for the custom model:**
- Add multiple attention heads (multi-head attention)
- Stack several attention layers with residual connections
- Add positional encodings so the model is aware of token order
- Train for more epochs or use a learning rate scheduler
- Add a feedforward sublayer after attention (as in a full Transformer block)